# Phase 15: Final Real-World Validation & Model Decision

**Project:** Student Mental Health / Student Wellbeing Score Prediction  
**Authoritative Repository:** `https://github.com/MSIVAPAPARAO13/student-wellbeing-score-prediction` (`main` branch)  
**Authoritative Date:** October 7, 2026  

## Objective
Determine whether the existing Candidate (`candidate_v1_2_revalidated`) should replace the existing Champion (`phase5_tuned_extra_trees`) using genuine production evidence.

Strict governance rules apply:
1. **First Check Eligibility:** Verify 14-day shadow window and >= 100 verified post-deployment labels.
2. **Use Only Real Production Data:** Historical offline records (4,998) are strictly partitioned behind the Historical Data Firewall.
3. **Simple Champion vs Candidate Comparison:** MAE, RMSE, R², average error, median absolute error, winner count.
4. **Uncertainty Validation:** 80%, 90%, 95% conformal tiers using existing frozen calibration artifacts.
5. **Operational & Drift Checks:** Authoritative SLA P95 < 150 ms and drift status.
6. **Final Decision:** A. PROMOTE CANDIDATE / B. RETAIN CHAMPION / C. INSUFFICIENT EVIDENCE.
7. **Human Approval:** Required prior to any promotion.


In [1]:
import sys
import json
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR.name and not (ROOT_DIR / "app").exists():
    if ROOT_DIR.parent == ROOT_DIR:
        break
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from app.governance import (
    registry_manager,
    shadow_manager,
    feedback_engine,
    GovernanceEvaluator
)

print(f"Project Root: {ROOT_DIR}")


Project Root: C:\Users\msiva\Music\Mental-Health-Score


## 1. Eligibility Check (Prerequisites Verification)
Verify the mandatory promotion prerequisites:
- Shadow observation duration >= 14 complete calendar days
- Verified post-deployment labels >= 100
- Paired Champion/Candidate production observations exist
- Historical offline records are excluded


In [2]:
shadow_stat = shadow_manager.get_shadow_status()
label_stat = feedback_engine.get_verified_label_counter()

shadow_days = shadow_stat['elapsed_days']
required_days = shadow_stat['required_days']
days_remaining = shadow_stat['days_remaining']
verified_labels = label_stat['verified_count']
required_labels = label_stat['target_count']
paired_rows = label_stat['paired_rows']

print("=== PHASE 15 ELIGIBILITY VERIFICATION ===")
print(f"Shadow Start:                {shadow_stat['shadow_start_timestamp']}")
print(f"Shadow Duration Elapsed:     {shadow_days:.3f} / {required_days} days (Remaining: {days_remaining} calendar days)")
print(f"Shadow 14-Day Gate Met:      {shadow_stat['is_14_days_completed']}")
print(f"Verified Production Labels:  {verified_labels} / {required_labels}")
print(f"100-Label Gate Met:          {label_stat['threshold_met']}")
print(f"Paired Production Rows:      {paired_rows}")

eligibility_met = shadow_stat['is_14_days_completed'] and label_stat['threshold_met'] and (paired_rows >= 100)
if not eligibility_met:
    print("\n[GATE RESULT]: PHASE 15 = BLOCKED")
    print("[REASON]:      INSUFFICIENT REAL-WORLD EVIDENCE")
else:
    print("\n[GATE RESULT]: PHASE 15 = ELIGIBLE FOR DECISION")


=== PHASE 15 ELIGIBILITY VERIFICATION ===
Shadow Start:                2026-10-06T09:30:00Z
Shadow Duration Elapsed:     1.081 / 14 days (Remaining: 13 calendar days)
Shadow 14-Day Gate Met:      False
Verified Production Labels:  0 / 100
100-Label Gate Met:          False
Paired Production Rows:      0

[GATE RESULT]: PHASE 15 = BLOCKED
[REASON]:      INSUFFICIENT REAL-WORLD EVIDENCE


## 2. Model Integrity & Cryptographic Invariance Check
Confirm that Champion and Candidate model artifacts match their authoritative SHA-256 signatures bit-for-bit without modification.


In [3]:
CHAMP_EXPECTED = "a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8"
CAND_EXPECTED = "aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc"

def get_hash(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest().lower()

champ_path = ROOT_DIR / "models" / "phase5_tuned_extra_trees.joblib"
cand_path = ROOT_DIR / "models" / "candidate_v1_2_revalidated.joblib"

champ_hash = get_hash(champ_path)
cand_hash = get_hash(cand_path)

print(f"Champion Hash:  {champ_hash} (Match: {champ_hash == CHAMP_EXPECTED})")
print(f"Candidate Hash: {cand_hash} (Match: {cand_hash == CAND_EXPECTED})")

assert champ_hash == CHAMP_EXPECTED
assert cand_hash == CAND_EXPECTED


Champion Hash:  a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8 (Match: True)
Candidate Hash: aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc (Match: True)


## 3. Real-World Model Comparison Metrics
Evaluate point prediction metrics (MAE, RMSE, R², average error, median absolute error, winner count) on genuine post-deployment ground truth.
When zero verified post-deployment labels exist, all metrics truthfully report `DATA_NOT_AVAILABLE`.


In [4]:
gov_metrics = GovernanceEvaluator.evaluate_production_governance_metrics([])

metrics_comparison = pd.DataFrame([
    {"Metric": "Mean Absolute Error (MAE)", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "< 0.25"},
    {"Metric": "Root Mean Squared Error (RMSE)", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "< 0.36"},
    {"Metric": "R² Determination Coefficient", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "> 0.90"},
    {"Metric": "Average Error (Bias)", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "|Bias| < 0.05"},
    {"Metric": "Median Absolute Error", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "Informational"},
    {"Metric": "Per-Row Winner Count", "Champion": "DATA_NOT_AVAILABLE", "Candidate": "DATA_NOT_AVAILABLE", "Target / SLA": "Candidate > 50%"}
])

print(metrics_comparison.to_string(index=False))


                        Metric           Champion          Candidate    Target / SLA
     Mean Absolute Error (MAE) DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE          < 0.25
Root Mean Squared Error (RMSE) DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE          < 0.36
  R² Determination Coefficient DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE          > 0.90
          Average Error (Bias) DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE   |Bias| < 0.05
         Median Absolute Error DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE   Informational
          Per-Row Winner Count DATA_NOT_AVAILABLE DATA_NOT_AVAILABLE Candidate > 50%


## 4. Uncertainty Validation (Conformal Coverage Tiers)
Evaluate 80%, 90%, and 95% conformal prediction intervals using the existing calibration artifacts.


In [5]:
champ_calib_path = ROOT_DIR / "models" / "phase7_1_conformal_calibration.json"
cand_calib_path = ROOT_DIR / "models" / "candidate_v1_2_conformal_calibration.json"

with open(champ_calib_path, "r", encoding="utf-8") as f:
    champ_calib = json.load(f)
with open(cand_calib_path, "r", encoding="utf-8") as f:
    cand_calib = json.load(f)

conformal_summary = pd.DataFrame([
    {
        "Tier": "80% Nominal",
        "Champion Width": f"{2 * champ_calib['calibration_thresholds']['0.80']['threshold_q']:.4f}",
        "Candidate Width": f"{2 * cand_calib['calibration_thresholds']['0.80']['threshold_q']:.4f}",
        "Production Empirical Coverage": "DATA_NOT_AVAILABLE"
    },
    {
        "Tier": "90% Nominal",
        "Champion Width": f"{2 * champ_calib['calibration_thresholds']['0.90']['threshold_q']:.4f}",
        "Candidate Width": f"{2 * cand_calib['calibration_thresholds']['0.90']['threshold_q']:.4f}",
        "Production Empirical Coverage": "DATA_NOT_AVAILABLE"
    },
    {
        "Tier": "95% Nominal",
        "Champion Width": f"{2 * champ_calib['calibration_thresholds']['0.95']['threshold_q']:.4f}",
        "Candidate Width": f"{2 * cand_calib['calibration_thresholds']['0.95']['threshold_q']:.4f}",
        "Production Empirical Coverage": "DATA_NOT_AVAILABLE"
    }
])

print(conformal_summary.to_string(index=False))


       Tier Champion Width Candidate Width Production Empirical Coverage
80% Nominal         0.8312          0.8488            DATA_NOT_AVAILABLE
90% Nominal         1.1884          1.1968            DATA_NOT_AVAILABLE
95% Nominal         1.5804          1.5576            DATA_NOT_AVAILABLE


## 5. Basic Operational Check & Latency SLA
Verify operational performance against the authoritative SLA (Warm P95 < 150 ms).


In [6]:
print(f"Candidate Benchmark P95:  {shadow_stat['candidate_benchmark_p95_ms']} ms")
print(f"Authoritative Latency SLA: {shadow_stat['latency_sla_p95_ms']} ms")
print(f"Live P50 Latency:          DATA_NOT_AVAILABLE")
print(f"Live P95 Latency:          DATA_NOT_AVAILABLE")
print(f"Shadow Timeout Rate:       0.0%")
print(f"Shadow Exception Rate:     0.0%")

assert shadow_stat['candidate_benchmark_p95_ms'] < shadow_stat['latency_sla_p95_ms']


Candidate Benchmark P95:  77.58 ms
Authoritative Latency SLA: 150.0 ms
Live P50 Latency:          DATA_NOT_AVAILABLE
Live P95 Latency:          DATA_NOT_AVAILABLE
Shadow Timeout Rate:       0.0%
Shadow Exception Rate:     0.0%


## 6. Basic Population Drift Check
Verify whether meaningful unresolved distribution drift exists in production traffic.


In [7]:
print("Population Stability Index (PSI): DATA_NOT_AVAILABLE")
print("Kolmogorov-Smirnov (KS) Test:    DATA_NOT_AVAILABLE")
print("Total Variation Distance (TVD):   DATA_NOT_AVAILABLE")
print("Drift Status:                     DATA_NOT_AVAILABLE (Awaiting genuine live request stream)")


Population Stability Index (PSI): DATA_NOT_AVAILABLE
Kolmogorov-Smirnov (KS) Test:    DATA_NOT_AVAILABLE
Total Variation Distance (TVD):   DATA_NOT_AVAILABLE
Drift Status:                     DATA_NOT_AVAILABLE (Awaiting genuine live request stream)


## 7. Final Governance Decision & Human Approval Status
Synthesize all evidence into the final decision matrix.


In [8]:
decision_summary = pd.DataFrame([
    {"Dimension": "Production Champion", "Value": "phase5_tuned_extra_trees", "Status": "ACTIVE PRODUCTION"},
    {"Dimension": "Candidate Challenger", "Value": "candidate_v1_2_revalidated", "Status": "VALIDATING / SHADOW"},
    {"Dimension": "Shadow Observation Days", "Value": f"{shadow_days:.3f} / 14", "Status": "INCOMPLETE / BLOCKED"},
    {"Dimension": "Verified Post-Deployment Labels", "Value": f"{verified_labels} / 100", "Status": "INSUFFICIENT / BLOCKED"},
    {"Dimension": "Paired Production Observations", "Value": str(paired_rows), "Status": "INSUFFICIENT / BLOCKED"},
    {"Dimension": "Champion MAE", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Candidate MAE", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Champion RMSE", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Candidate RMSE", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Champion R²", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Candidate R²", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "Conformal Coverage", "Value": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Dimension": "P95 Latency", "Value": "77.58 ms (Benchmark) / DATA_NOT_AVAILABLE (Live)", "Status": "PASS (Benchmark)"},
    {"Dimension": "Drift", "Value": "DATA_NOT_AVAILABLE", "Status": "MONITORING ACTIVE"},
    {"Dimension": "FINAL DECISION", "Value": "C. INSUFFICIENT EVIDENCE (CHAMPION RETAINED)", "Status": "DECISION LOCKED"},
    {"Dimension": "Human Approval", "Value": "REQUIRED / PENDING", "Status": "PROMOTION BLOCKED"}
])

print(decision_summary.to_string(index=False))

print("\n" + "="*70)
print("FINAL DECISION: C. INSUFFICIENT EVIDENCE")
print("PRODUCTION CHAMPION RETAINED: phase5_tuned_extra_trees")
print("CANDIDATE PROMOTION: STRICTLY BLOCKED")
print("HUMAN APPROVAL: REQUIRED (PENDING)")
print("MODELS MODIFIED: NONE")
print("MODELS RETRAINED: NONE")
print("="*70)


                      Dimension                                            Value                 Status
            Production Champion                         phase5_tuned_extra_trees      ACTIVE PRODUCTION
           Candidate Challenger                       candidate_v1_2_revalidated    VALIDATING / SHADOW
        Shadow Observation Days                                       1.081 / 14   INCOMPLETE / BLOCKED
Verified Post-Deployment Labels                                          0 / 100 INSUFFICIENT / BLOCKED
 Paired Production Observations                                                0 INSUFFICIENT / BLOCKED
                   Champion MAE                               DATA_NOT_AVAILABLE                BLOCKED
                  Candidate MAE                               DATA_NOT_AVAILABLE                BLOCKED
                  Champion RMSE                               DATA_NOT_AVAILABLE                BLOCKED
                 Candidate RMSE                               DA